# 11. 代价核算（预测篇）

## 11.1 目的与方法

本章只做**预测**，不测任何东西。把第 2 到 9 章的局部估算合并成一张总表，产出：

1. 训练一步的 FLOPs 拆分；
2. 显存的预估；
3. 一份"预测清单"（11.5），第 12 章实测后逐项对照。

整个 Part 1 的方法是闭环：**预测（第 11 章）→ 测量（第 12 章）→ 解释差异**。先写下预测再去测，才能知道哪里"出乎意料"。

## 11.2 模块级核算总表

符号同第 1 章，$N=B\cdot T$，$d_{ff}=4d$。

| 模块（出现次数） | 参数 | FLOPs（forward） | 为反向保存的张量（元素数） | 类型 | kernel 数（forward，估） |
|---|---|---|---|---|---|
| Embedding（1） | $Vd$ | 0 | 索引 $N$ | 访存 | 1 |
| RMSNorm（$2L+1$） | $d$ | 约 $5Nd$ | $Nd$ 加 $N$ | 访存 | 约 6 |
| $q,k,v,o$ 投影（$4L$） | $d^2$ 各 | $2Nd^2$ 各 | 输入 $Nd$ | 计算 | 1 各 |
| RoPE（$2L$ 次调用） | 0 | 约 $3Nd$ 每次 | 中间量约 $Nd$ | 访存 | 约 7 每次 |
| $QK^\top$（$L$） | 0 | $2BT^2d$ | $Q,K$ | 计算，强度偏低 | 1 |
| 缩放、mask、softmax（$L$） | 0 | $O(BhT^2)$ | $c\cdot BhT^2$ | 访存 | 约 7 |
| $PV$（$L$） | 0 | $2BT^2d$ | $P,V$ | 计算，强度偏低 | 1 |
| SwiGLU 三个 GEMM（$L$） | $3dd_{ff}$ | $6Ndd_{ff}$ | $5Nd_{ff}$ 加 $Nd$ | 计算 | 3 |
| SwiGLU 逐元素（$L$） | 0 | $O(Nd_{ff})$ | 含在上一行 | 访存 | 3 |
| 残差加法（$2L$） | 0 | $Nd$ 各 | 无 | 访存 | 1 各 |
| `lm_head`（1） | $Vd$ | $2NdV$ | $Nd$ | 计算 | 1 |
| 交叉熵（1） | 0 | $O(NV)$ | $NV$ 的若干份 | 访存 | 数个 |

数字出处：第 4 到 9 章。"kernel 数"和"保存份数"是估计，第 12 章实测后替换。

## 11.3 训练一步的 FLOPs 拆分

**forward**（每个 token）：

$$
\underbrace{8d^2L}_{\text{注意力投影}}+\underbrace{24d^2L}_{\text{FFN}}+\underbrace{4TdL}_{\text{注意力 matmul}}+\underbrace{2dV}_{\texttt{lm\_head}}
$$

训练一步为 forward 的 3 倍（backward 约 2 倍，见第 6、7 章）。不含 optimizer，也不含逐元素运算。

```python
def flops_breakdown(d, L, V, B, T, d_ff=None):
    d_ff = d_ff or 4 * d; N = B * T
    f = {"attn_proj": L * 8 * N * d * d, "ffn": L * 6 * N * d * d_ff,
         "attn_matmul": L * 4 * N * T * d, "lm_head": 2 * N * d * V}
    tot = sum(f.values())
    return {**f, "fwd_total": tot, "step_total": 3 * tot}
```

**small，$B=4$，$T=512$**：

| 项 | forward (GFLOP) | 占比 |
|---|---|---|
| 注意力投影 | 115.9 | 21.7% |
| FFN | 347.9 | 65.2% |
| 注意力 matmul | 38.7 | 7.2% |
| `lm_head` | 31.5 | 5.9% |
| 合计 | 534.0 | 100% |

训练一步约 1.60 TFLOP，与 Step 2 一致。用 `flops_breakdown` 画一张堆叠柱状图（横轴 $T$，纵轴占比）。

## 11.4 ctx 增大时注意力占比为什么上升

线性项（投影、FFN、`lm_head`）与 $N$ 成正比，注意力 matmul 与 $N\cdot T$ 成正比。二者之比（$d_{ff}=4d$）：

$$
\frac{\text{注意力 matmul}}{\text{层内线性}}=\frac{4NTd}{32Nd^2}=\frac{T}{8d}
$$

**注意力 matmul 占 forward FLOPs 的比例**（$B$ 不影响占比）：

| 模型 | $T=128$ | $T=512$ | $T=1024$ | $T=4096$ |
|---|---|---|---|---|
| small（$d=768$） | 1.9% | 7.2% | 13.5% | 38.4% |
| large（$d=1280$） | 1.2% | 4.7% | 9.0% | 28.3% |

宽度越大占比越低（分母里的 $d^2$ 比分子的 $d$ 增长更快）。

**FLOPs 占比不等于时间占比。** 注意力里的缩放、mask、softmax 是 $T^2$ 大小的访存密集运算（第 6 章），按第 8 章的估算，small 在 $T=512$ 时注意力核心约占一个 block 时间的三分之一，而 FLOPs 只占约 7%。这是本章最有价值的结论，由第 12 章的 nsys 验证。

## 11.5 预测清单（第 12 章回填的依据）

以下均为 small，$B=4$，$T=512$，fp32，单卡。

**(1) 阶段时间**

| 阶段 | FLOPs | $P$（TFLOP/s） | 理想时间 | 访存部分 | 预测总时间 |
|---|---|---|---|---|---|
| forward | 0.534 T | 33.4 | 16.0 ms | 约 20 ms | 约 37 ms |
| backward | 1.068 T | 33.4 | 32.0 ms | 取 forward 的约 2 倍作粗估 | 待估 |
| optimizer | 28 B/参数 | BW 769 GB/s | 4.7 ms（下限） | | 见下 |

optimizer 的 4.7 ms 是带宽下限。staff 的 AdamW 若是逐参数 Python 循环，实际会被 kernel 发射开销主导，预期明显高于下限（12.9 解释）。

**(2) 时间占比**（FLOPs 占比与预测时间占比并排）

| 项 | FLOPs 占比 | 预测时间占比 |
|---|---|---|
| 注意力（matmul + 缩放 + mask + softmax） | 7.2% | 约 35%（第 8 章） |
| FFN（GEMM + 逐元素） | 65.2% | 约 40% |
| 其余（投影、norm、RoPE、残差、`lm_head`、loss） | 约 27.6% | 约 25%（余量） |

**(3) forward kernel 数与时间，按类别**

| 类别 | 预测 kernel 数 | 预测时间占比 |
|---|---|---|
| gemm | 约 109（$(7L+1)$ 个线性层加 $2L$ 个注意力 matmul） | 约 45% |
| elementwise | 约 300 | 约 40% |
| reduce | 约 25 | 约 3% |
| softmax | 12 | 约 5% |
| copy / cat | 约 40 | 约 3% |

这张表是按第 8.9 的 kernel 清单粗略相加的，误差较大，实测后直接替换。

**(4) forward 时间，按区间**（沿用第 8.9、9.9）

| 区间 | 预测 kernel 数 | 预测时间 (ms) |
|---|---|---|
| embed | 1 | 0.02 |
| block（×12） | 约 600（每个约 50） | 34.8 |
| ln_final | 6 | 0.09 |
| lm_head | 1 | 0.94 |
| loss | 数个 | 1.1 |

## 11.6 显存预估

| 部分 | 估算 |
|---|---|
| 权重 + 梯度 + Adam | $16P$ B（fp32，$P$ 为参数个数） |
| FFN 激活 | $5Nd_{ff}$ 每层 |
| 注意力分数 | $c\,BhT^2$ 每层，$c\approx1$ 到 3 |
| 其余 $N\times d$ 张量 | 约 15 到 25 份每层 |
| logits 与损失 | $NV$ 的约 3 到 4 份 |

```python
def est_peak_gib(P, d, L, h, V, B, T, c=2, other_per_layer=20):
    N = B * T
    static = 16 * P
    act = L * 4 * (5 * N * 4 * d + c * B * h * T * T + other_per_layer * N * d) + 4 * 4 * N * V
    return (static + act) / 2**30
```

实测与对照在 12.10。

---


# 12. 测量：benchmark、NVTX、nsys 与 ncu，并回填第 11 章的预测

## 12.1 分工与流程

| 工具 | 回答什么 | 注意 |
|---|---|---|
| `benchmark.py`（不加 profiler） | **多快**：各阶段的真实耗时 | 绝对时间只信它 |
| nsys（Nsight Systems） | **时间花在哪**：kernel 种类、次数、占比、空隙、CPU 发射 | 有开销，只看构成，不看绝对时间 |
| ncu（Nsight Compute） | **单个 kernel 为什么慢**：算力还是带宽受限、占用率、访存效率 | 每个 kernel 被重放多次，只抓少数几个；duration 不可信，只看指标与比例 |
| `torch.profiler` | 算子（`aten::`）级的 CUDA 时间 | 与 nsys 互补 |
| 显存快照与 `max_memory_allocated` | 显存峰值与构成 | 对照 11.6 |

流程是从粗到细：

1. 12.2 测真实耗时；
2. 12.3 到 12.9 用 nsys 得到构成，找出值得深挖的 kernel；
3. 12.10 用 ncu 深挖这几个 kernel，验证"为什么"；
4. 12.11 补算子级与显存；
5. 12.12 到 12.14 避坑、取数、回填并解释。

## 12.2 先测真实耗时（`benchmark.py`）

GPU 异步，计时必须同步；首次迭代含 cuBLAS 初始化与懒加载，必须预热。

```python
import torch, statistics

def bench(fn, warmup=5, iters=20):
    for _ in range(warmup): fn()
    torch.cuda.synchronize()
    ts = []
    for _ in range(iters):
        s, e = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
        s.record(); fn(); e.record(); torch.cuda.synchronize()
        ts.append(s.elapsed_time(e))              # ms
    return statistics.median(ts), min(ts), max(ts)
```

分别计时 `forward`（`no_grad` 下，才能与 11.5 的 forward 预测对比）、`forward+loss+backward`、完整一步（含 optimizer）。取中位数，同时看最小与最大，抖动大说明卡不空闲或有其他进程。


- 结果如下：

```bash
CUDA_VISIBLE_DEVICES=0 uv run python scripts/benchmark.py     --model small --ctx 512 --batch 4 --dtype float32 --mode all
device: NVIDIA GeForce RTX 5070 Ti  torch 2.11.0+cu130
model=small params=128.6M ctx=512 batch=4 dtype=float32 compile=False
forward FLOPs = 534.0 GFLOP  (step = 3x = 1.60 TFLOP)
mode        median      min      max  jitter     wall     tok/s  TFLOP/s  peak GiB   (ms)
fwd          37.72    37.40    37.76     1%    37.73     54275    14.16      0.71
fwdbwd      114.90   114.72   115.16     0%   114.71     17854    13.94      4.06
full        127.85   127.70   128.04     0%   127.72     16035    12.53      5.04
saved -> results/bench/bench_small_ctx512_b4_float32.json
```

## 12.3 NVTX 基础

**它是什么**：在 CPU 侧给一段代码打上带名字的区间，nsys 在时间线上显示。GPU 是异步的，所以：

| 现象 | 说明 |
|---|---|
| 区间里的 PyTorch 调用只是**发射** kernel | 若区间结束时没有 `synchronize`，CPU 区间很短，GPU 工作还在后面 |
| nsys 把 kernel **投影**回发射它的 NVTX 区间 | 这才是"该区间对应的 GPU 工作" |
| 投影得到的是 kernel 的**时间跨度**（含 kernel 之间的空隙），不是耗时之和 | 要耗时之和，用按区间汇总 kernel 的报告（12.7） |

**API**（`torch.cuda.nvtx`）：

```python
import torch.cuda.nvtx as nvtx
with nvtx.range("forward"):       # 推荐，自动配对
    ...
nvtx.range_push("x"); ...; nvtx.range_pop()   # 手动配对，跨函数时用
nvtx.mark("checkpoint")                        # 时间线上的瞬时标记
```

**命名规范**：

- 用**静态字符串**。nsys 按名字汇总，名字里带 step 编号会让每个 step 变成一行。
- 层级用点号：`attn.scores`、`attn.softmax`、`attn.av`。
- 同一类层用**同一个名字**（如 `block`），汇总时调用次数就是层数，便于自检（每个 forward 应有 $L$ 次）。
- 需要看单个层时临时加编号，不作为默认。

**开销**：每个区间是微秒量级的 CPU 开销。细标注会让 CPU 变慢，对小模型、短 ctx（本来就发射受限）影响尤其大。所以粗粒度用于计时，细粒度只在 profile 运行里开。

**NVTX 同时服务于 ncu**：ncu 也可以按 NVTX 区间过滤 kernel（12.10），所以这里的标注一次打好，两个工具共用。

## 12.4 本项目的标注方案

| 区间 | 内容 | 位置 |
|---|---|---|
| `step` | 一次完整训练步 | 最外层 |
| `forward` | embedding 到 logits | 主线程 |
| `loss` | 交叉熵 | 主线程 |
| `backward` | `loss.backward()` | 见 12.8，**归因有特殊问题** |
| `optimizer` | `opt.step()` 与 `zero_grad` | 主线程 |
| `embed`、`block`、`ln_final`、`lm_head` | forward 内部 | forward hook |
| `ln1`、`attn`、`ln2`、`ffn` | block 内部 | forward hook |
| `attn.scores`、`attn.softmax`、`attn.av` | attention 内部三段 | 替换函数（作业要求） |

**attention 内部三段**：把 staff 的 `scaled_dot_product_attention` 换成带标注的版本。`CausalMultiHeadSelfAttention.forward` 调用时按模块全局名查找该函数，所以替换模块属性即可，不改 staff 源码：

```python
import math, torch, torch.cuda.nvtx as nvtx
from einops import einsum
import cs336_basics.model as M
from cs336_basics.nn_utils import softmax

def annotated_sdpa(Q, K, V, mask=None):
    d_k = K.shape[-1]
    with nvtx.range("attn.scores"):                     # QK^T、缩放、mask
        s = einsum(Q, K, "... q d, ... k d -> ... q k") / math.sqrt(d_k)
        if mask is not None:
            s = torch.where(mask, s, float("-inf"))
    with nvtx.range("attn.softmax"):
        w = softmax(s, dim=-1)
    with nvtx.range("attn.av"):
        return einsum(w, V, "... q k, ... k d -> ... q d")

M.scaled_dot_product_attention = annotated_sdpa          # 只在 profile 运行里替换
```

**模块级标注**（hook，不改源码）：

```python
def add_nvtx_hooks(model, names=("RMSNorm", "CausalMultiHeadSelfAttention", "SwiGLU",
                                 "TransformerBlock", "Embedding", "Linear")):
    hs = []
    for mod in model.modules():
        k = type(mod).__name__
        if k in names:
            hs.append(mod.register_forward_pre_hook(lambda m, a, k=k: nvtx.range_push(k)))
            hs.append(mod.register_forward_hook(lambda m, a, o: nvtx.range_pop()))
    return hs
```

区间名用类名，所以所有 `RMSNorm` 汇总成一行，调用次数应为 $2L+1$，是很好的自检（`Linear` 含 `lm_head`，应为 $7L+1$）。forward hook 只覆盖 forward，不覆盖反向。

## 12.5 profile 运行脚本（骨架）`scripts/profile_run.py`

```python
def phase(name, fn):
    with nvtx.range(name):
        out = fn()
        torch.cuda.synchronize()      # 每个阶段末尾同步，GPU 工作才落在对应区间的时间窗内
    return out

for _ in range(args.warmup): train_step()     # 预热，不被抓取
torch.cuda.synchronize()

torch.cuda.profiler.start()                    # 只从这里开始抓取
for _ in range(args.steps):                    # nsys 3 步足够，多了文件很大
    with nvtx.range("step"):
        loss = phase("forward", fwd_and_loss)
        phase("backward", loss.backward)
        phase("optimizer", opt_step_and_zero)
torch.cuda.synchronize()
torch.cuda.profiler.stop()
```

参数：`--model --ctx --batch --dtype --mode {fwd,fwdbwd,full} --warmup --steps --nvtx-fine`。`--nvtx-fine` 打开 12.4 的替换与 hook。

每个阶段末尾同步会破坏阶段之间的重叠，所以得到的是"各阶段互不重叠"的构成，不是真实流水线。这对归因有利，对测速不利，所以 12.1 把二者分开。

## 12.6 运行 nsys

```bash
cd /data2/c220/hz/cs336
mkdir -p results/nsys
CUDA_VISIBLE_DEVICES=0 uv run nsys profile \
  -o results/nsys/small_ctx512_fp32 --force-overwrite=true \
  --trace=cuda,nvtx,osrt \
  --capture-range=cudaProfilerApi --capture-range-end=stop \
  python scripts/profile_run.py --model small --ctx 512 --batch 4 --mode full \
         --warmup 5 --steps 3 --nvtx-fine
```

| 选项 | 作用 |
|---|---|
| `--trace=cuda,nvtx,osrt` | 抓 CUDA API 与 kernel、NVTX、OS 运行时 |
| `--capture-range=cudaProfilerApi` 加 `--capture-range-end=stop` | **只抓 `start()` 到 `stop()` 之间**，预热不进文件 |
| `--force-overwrite=true` | 覆盖同名文件 |
| `--cuda-memory-usage=true`（可选） | 记录显存分配时间线，有额外开销 |

输出文件名带上配置（模型、ctx、dtype）。开始前 `nvidia-smi` 确认卡是空的，另一张卡用 `CUDA_VISIBLE_DEVICES` 隔离。

## 12.7 读 nsys 结果

**命令行**。报告名随版本变化，先 `nsys stats --help-reports` 看本机有哪些：

| 想看什么 | 报告（常见名字） |
|---|---|
| 全部 kernel 的汇总（次数、总时间、平均） | `cuda_gpu_kern_sum` |
| 每个 NVTX 区间的 CPU 时间 | `nvtx_sum` |
| 每个 NVTX 区间投影到 GPU 的**时间跨度** | `nvtx_gpu_proj_sum` |
| **按 NVTX 区间汇总 kernel**（耗时之和） | `nvtx_kern_sum` |
| 拷贝时间与大小 | `cuda_gpu_mem_time_sum`、`cuda_gpu_mem_size_sum` |
| CPU 侧 CUDA API 耗时（发射开销） | `cuda_api_sum` |
| 逐次 kernel 明细 | `cuda_gpu_trace` |

```bash
uv run nsys stats --report cuda_gpu_kern_sum --report nvtx_sum \
     --format csv --output results/nsys/small_ctx512 results/nsys/small_ctx512_fp32.nsys-rep
```

CSV 的文件名和列名以实际输出为准，用 `glob` 找文件，用 `df.columns` 看列。

**汇总成类别表**（供 11.5(3) 回填）：

In [1]:
import pandas as pd

k = pd.read_csv("../results/nsys/small_ctx512_cuda_gpu_kern_sum.csv")
n = pd.read_csv("../results/nsys/small_ctx512_nvtx_sum.csv")
k.columns = [c.strip() for c in k.columns]

print(n[["Range", "Instances", "Total Time (ns)"]].sort_values("Instances", ascending=False))
print(k[["Time (%)", "Total Time (ns)", "Instances", "Name"]].head(20))


                            Range  Instances  Total Time (ns)
6                         :Linear        255         15884589
8                        :RMSNorm         75          7917519
9                    :attn.scores         36          4290846
7                         :SwiGLU         36          9523969
4               :TransformerBlock         36         53846153
5   :CausalMultiHeadSelfAttention         36         34375424
10                  :attn.softmax         36          2957745
11                       :attn.av         36          2947591
0                           :step          3        410887560
1                       :backward          3        230439187
2                        :forward          3        117147989
3                      :optimizer          3         63241743
12                     :Embedding          3           277409
13      CCCL:cub::DeviceRadixSort          3            30532
    Time (%)  Total Time (ns)  Instances  \
0       16.0         58322

- 第二步：不写 SQL，先要"每阶段的 kernel 构成"

nsys 有现成报告（2024.6 可用；名字随版本会变，`nsys stats --help-nvtx-reports` 可查）：
```bash
uv run nsys stats --report nvtx_kern_sum \
    --format csv --output results/nsys/xkern results/nsys/small_ctx512_fp32.nsys-rep
uv run nsys stats --report nvtx_gpu_proj_sum \
    --format csv --output results/nsys/xproj results/nsys/small_ctx512_fp32.nsys-rep
```
- `nvtx_kern_sum`：kernel 按 **launch 侧 NVTX 区间**分组汇总（最内层归属）。
- `nvtx_gpu_proj_sum`：把 NVTX 区间**投影到 GPU 时间线**再汇总——异步重叠会被正确摊开，最接近"官方版方案 A"。
两个都跑一下互相对照；如果 `nvtx_gpu_proj_sum` 里 backward 的 GPU 时间和 `nvtx_sum` 里 backward 窗口的 `Total` 差距很小，说明你的 sync 生效了、窗口≈执行窗。


In [2]:
import pandas as pd, glob, re
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)

def load(pattern):
    d = pd.read_csv(glob.glob(pattern)[0])
    d.columns = [c.strip() for c in d.columns]
    return d

kern = load("../results/nsys/xkern*.csv")     # nvtx_kern_sum
proj = load("../results/nsys/xproj*.csv")     # nvtx_gpu_proj_sum
print(kern.columns.tolist())
print(proj.columns.tolist())

# 把 kernel 名压短
def short(n, w=48):
    n = re.sub(r"<.*>", "<…>", str(n))         # 去掉模板参数
    n = re.sub(r"^void\s+", "", n)
    return n[:w]

RANGE, TOTAL, INST, NAME = "NVTX Range", "Total Time (ns)", "Kern Inst", "Kernel Name"   # ← 按实际列名改
STEPS = 3

# 表 A：每个区间的 kernel 总数、总时间（每步平均）
a = (kern.groupby(RANGE)
         .agg(kernels=(INST, "sum"), total_ms=(TOTAL, lambda s: s.sum() / 1e6))
         .assign(kernels_per_step=lambda t: t.kernels / STEPS,
                 ms_per_step=lambda t: t.total_ms / STEPS)
         .sort_values("total_ms", ascending=False))
print(a.round(2))

# 表 B：某个区间里最大的几类 kernel，及其在该区间内的占比
def top_in(range_name, n=8):
    d = kern[kern[RANGE] == range_name].copy()
    d["kernel"] = d[NAME].map(short)
    d["share_%"] = d[TOTAL] / d[TOTAL].sum() * 100
    d["per_step"] = d[INST] / STEPS
    return (d[["kernel", "per_step", "share_%"]]
              .sort_values("share_%", ascending=False).head(n).round(1))

print(top_in("forward"))
print(top_in("optimizer"))

RANGE_P, TOTAL_P, INST_P = "Range", "Total Range Time (ns)", "Range Instances"   # ← 按实际列名改
p = (proj[[RANGE_P, INST_P, TOTAL_P]]
        .assign(ms_per_inst=lambda t: t[TOTAL_P] / t[INST_P] / 1e6)
        .sort_values(TOTAL_P, ascending=False))
print(p.round(3))

['NVTX Range', 'Style', 'PID', 'TID', 'NVTX Inst', 'Kern Inst', 'Total Time (ns)', 'Avg (ns)', 'Med (ns)', 'Min (ns)', 'Max (ns)', 'StdDev (ns)', 'Kernel Name']
['Range', 'Style', 'Total Proj Time (ns)', 'Total Range Time (ns)', 'Range Instances', 'Proj Avg (ns)', 'Proj Med (ns)', 'Proj Min (ns)', 'Proj Max (ns)', 'Proj StdDev (ns)', 'Total GPU Ops', 'Avg GPU Ops', 'Avg Range Lvl', 'Avg Num Child']
                               kernels  total_ms  kernels_per_step  ms_per_step
NVTX Range                                                                     
:step                             7254    143.32            2418.0        47.77
:forward                          1923    112.11             641.0        37.37
:TransformerBlock                 1872    105.08             624.0        35.03
:Linear                            255     65.84              85.0        21.95
:CausalMultiHeadSelfAttention     1152     52.65             384.0        17.55
:SwiGLU                            216

In [3]:
# 用 TID 验证线程假设（两行）：
print(kern.groupby("NVTX Range")["TID"].agg(lambda s: sorted(set(s))))
print(kern[kern["NVTX Range"] == ":backward"][["Kern Inst", "Total Time (ns)", "Kernel Name"]])

# 用差分法估算 backward 的总量（不分区间，用总表）：
ks = pd.read_csv("../results/nsys/small_ctx512_cuda_gpu_kern_sum.csv")
ks.columns = [c.strip() for c in ks.columns]
tot_n  = ks["Instances"].sum() / 3
tot_ms = ks["Total Time (ns)"].sum() / 1e6 / 3
bwd_n  = tot_n  - 641 - 1776           # backward 的 kernel 数/步（估算）
bwd_ms = tot_ms - 37.37 - 10.40        # backward 的 GPU 时间/步（估算）
print(bwd_n, bwd_ms, bwd_ms / 76.81)   # 最后一项是 backward 的 GPU 利用率

NVTX Range
:CausalMultiHeadSelfAttention    [1106692]
:Embedding                       [1106692]
:Linear                          [1106692]
:RMSNorm                         [1106692]
:SwiGLU                          [1106692]
:TransformerBlock                [1106692]
:attn.av                         [1106692]
:attn.scores                     [1106692]
:attn.softmax                    [1106692]
:backward                        [1106692]
:forward                         [1106692]
:optimizer                       [1106692]
:step                            [1106692]
CCCL:cub::DeviceRadixSort        [1106790]
Name: TID, dtype: object
    Kern Inst  Total Time (ns)                                                  Kernel Name
97          3             2208  void at::native::vectorized_elementwise_kernel<(int)4, a...
1238.0 73.84702133333334 0.9614245714533699


In [4]:
ks = pd.read_csv("../results/nsys/small_ctx512_cuda_gpu_kern_sum.csv")
ks.columns = [c.strip() for c in ks.columns]
tot = ks.set_index("Name")[["Instances", "Total Time (ns)"]].rename(
        columns={"Instances": "n", "Total Time (ns)": "t"})

def per_kernel(r):
    d = kern[kern["NVTX Range"] == r]
    return d.groupby("Kernel Name").agg(n=("Kern Inst", "sum"), t=("Total Time (ns)", "sum"))

fwd, opt = per_kernel(":forward"), per_kernel(":optimizer")
bwd = tot.sub(fwd, fill_value=0).sub(opt, fill_value=0)

print("有负值（名字没对上）:", (bwd < 0).any().any())
print("backward kernel 数/步:", bwd.n.sum() / 3)       # 应约 1238

import re
CATS = [("gemm", r"gemm|xmma|cutlass|cublas"), ("softmax", r"softmax"),
        ("reduce", r"reduce"), ("elementwise", r"elementwise|vectorized"),
        ("copy/cat", r"copy|cat|memcpy"), ("index/embed", r"index|embedding|scatter|gather|sort"),
        ("optimizer", r"multi_tensor|adam")]
def cat(n):
    for c, pat in CATS:
        if re.search(pat, n, re.I): return c
    return "other"

def table(d):
    d = d[d.n > 0].copy()
    d["cat"] = [cat(x) for x in d.index]
    g = d.groupby("cat")[["n", "t"]].sum()
    g["n/step"] = g.n / 3
    g["ms/step"] = g.t / 1e6 / 3
    g["share_%"] = g.t / g.t.sum() * 100
    return g[["n/step", "ms/step", "share_%"]].sort_values("ms/step", ascending=False).round(2)

print("forward:\n", table(fwd))
print("backward:\n", table(bwd))
print("optimizer:\n", table(opt))

有负值（名字没对上）: False
backward kernel 数/步: 1238.0
forward:
              n/step  ms/step  share_%
cat                                  
gemm          109.0    24.33    65.11
elementwise   456.0    11.32    30.28
reduce         52.0     1.42     3.80
copy/cat       24.0     0.30     0.81
backward:
              n/step  ms/step  share_%
cat                                  
gemm          218.0    39.11    52.97
elementwise   929.0    32.67    44.25
reduce         77.0     1.89     2.56
copy/cat       12.0     0.16     0.21
index/embed     2.0     0.01     0.02
optimizer:
              n/step  ms/step  share_%
cat                                  
elementwise  1776.0     10.4    100.0


In [5]:
pd.set_option("display.max_colwidth", 150)
def full(n, w=150): return re.sub(r"^void\s+", "", str(n))[:w]

def top_by_cat(d, category, n=6):
    d = d[d.n > 0].copy()
    d["cat"] = [cat(x) for x in d.index]
    d = d[d.cat == category].copy()
    d["n/step"]  = d.n / 3
    d["avg_us"]  = d.t / d.n / 1e3
    d["ms/step"] = d.t / 1e6 / 3
    d["kernel"]  = [full(x) for x in d.index]
    return (d[["n/step", "avg_us", "ms/step", "kernel"]]
              .sort_values("ms/step", ascending=False).head(n).round(2))

print(top_by_cat(fwd, "gemm"));        print(top_by_cat(bwd, "gemm"))
print(top_by_cat(fwd, "elementwise")); print(top_by_cat(bwd, "elementwise"))

d = kern[kern["NVTX Range"] == ":attn.softmax"]
print(d.assign(name=d["Kernel Name"].map(full))[["Kern Inst", "Avg (ns)", "name"]])

                                                                                                                                                        n/step  avg_us  ms/step  \
Kernel Name                                                                                                                                                                       
void magma_sgemmEx_kernel<float, float, float, (bool)1, (bool)0, (int)6, (int)4, (int)6, (int)3, (int)4>(int, int, int, BatchedTensor, int, Batched...    85.0  213.54    18.15   
void cutlass::Kernel2<cutlass_80_simt_sgemm_128x256_8x4_tn_align1>(T1::Params)                                                                            12.0  422.78     5.07   
void cutlass::Kernel2<cutlass_80_simt_sgemm_64x64_8x5_nn_align1>(T1::Params)                                                                              12.0   92.24     1.11   

                                                                                                        


- 第三步：GUI 过一遍时间线
```bash
nsys-ui results/nsys/small_ctx512_fp32.nsys-rep
```
按这张清单看（比看数字更快发现标注问题）：
1. **NVTX 行**：应看到 3 组完全重复的 `step → forward → [模块嵌套] → attn.*` + `backward` + `optimizer`。三步长得不一样 → 步间 kernel 序列不稳定（回头查 zero_grad）。
2. **对齐**：kernel 行上的小方块应基本落在对应 phase 条**内部**；forward 条右侧溢出一串 kernel → 说明抓的那次没 sync。
3. **Memory / memcpy 行**：窗口内应几乎为空（数据常驻 GPU + `set_to_none`）。有 H2D/memset 混进 forward/backward → 有意外拷贝。
4. **悬停任意 kernel**：弹窗里能看到它的 NVTX 父链（如 step→forward→TransformerBlock→Linear），抽查几个 GEMM 归属是否正确。
5. **宽度直觉**：backward 条 ≈ 2× forward 条、optimizer 条窄——和第 11 章预测定性一致即可。


- 第四步：SQL 精细查询（可选做）

sqlite 已经生成，直接用。先探 schema（版本间列名略有差异），再做三个标准查询：


In [6]:

# import sqlite3, pandas as pd
# con = sqlite3.connect("../results/nsys/small_ctx512_fp32.sqlite")
# # print([r[1] for r in con.execute("PRAGMA table_info(NVTX_EVENTS)")])          # 首次先看一眼
# # print([r[1] for r in con.execute("PRAGMA table_info(CUPTI_ACTIVITY_KIND_KERNEL)")])
# nvtx = pd.read_sql("""
#     SELECT e.start, e.end, s.value AS name
#     FROM NVTX_EVENTS e LEFT JOIN StringIds s ON e.text = s.id""", con)
# ker = pd.read_sql("""
#     SELECT k.start, k.end, kn.value AS name
#     FROM CUPTI_ACTIVITY_KIND_KERNEL k
#     LEFT JOIN StringIds kn ON k.demangledName = kn.id""", con)
# def window_kernels(name):
#     rows = []
#     for _, w in nvtx[nvtx.name == name].iterrows():
#         m = ker[(ker.start >= w.start) & (ker.end <= w.end)]   # 包含式：sync 保证无跨界
#         rows.append(m)
#     return rows
# bwins = window_kernels("backward")

# # **查询 1 —— backward 的 kernel 构成（方案 A 的产出）**：

# def classify(n):
#     l = n.lower()
#     if "gemm" in l or "cutlass" in l:          return "GEMM"
#     if "softmax" in l:                          return "softmax"
#     if "elementwise" in l or "vectorized" in l: return "elementwise"
#     if "reduce" in l:                           return "reduce"
#     if "copy" in l or "cat" in l or "index" in l: return "copy/index"
#     return "other"
# m = bwins[0].copy()
# m["cat"] = m["name"].map(classify)
# g = m.groupby("cat").agg(n_kernels=("name", "size"),
#                          gpu_ms=("name", lambda _: (m.end - m.start).sum() / 1e6))

# #（按窗口分组求和写严谨一点即可；这里示意。)
# # 预期：backward 里 GEMM 数量 ≈ forward 的 2 倍（每个 Linear 权重和输入各求一次导），elementwise 一堆小 kernel。

# # **查询 2 —— 步间一致性**（呼应 `set_to_none` 那段讨论）：

# seqs = [list(w.sort_values("start")["name"]) for w in bwins]
# print("三步 backward kernel 序列完全一致:", seqs[0] == seqs[1] == seqs[2])

# # 不一致 → 第 2 步起混进了额外 kernel（典型原因：grad 没清干净或优化器 state 懒分配漏出 capture range）。

# # **查询 3 —— 窗口质量**：每步 `backward` 窗口时长 vs 窗内 kernel GPU 时间之和，比值接近 1 说明窗口紧实、方案 A 的归因边界可靠。


- 第五步：解读纪律 + 去向

从 CSV/SQL 里得到的**结论类型只有一种：构成**。哪个 kernel 类、多少次、占多少比例；forward:backward 的 GEMM 比例是否符合 1:2 量级；attention 三段（scores/softmax/av）谁是大头。
**绝对时长、阶段占比的定量结论回到 benchmark.py**——本脚本的 sync 已使时序失真。
若发现某类 kernel 占比异常（比如 elementwise 吃掉一半、或 softmax 意外的大），那就是下一章优化的靶子。
之后自然衔接 ncu：
```bash
uv run ncu --profile-from-start off --nvtx-include "forward/" \
    -k "regex:sgemm|cutlass" -c 1 --set detailed \
    python scripts/profile_run.py --model small --ctx 512 --batch 4 \
        --mode fwd --warmup 2 --steps 1
```
先只看 forward 的头号 GEMM 离 roofline 多远，再逐步放开。



- 小结这四步的递进

| 步 | 工具 | 得到什么 |
|---|---|---|
| 1 | 两张 CSV | 标注自检 + kernel 总排行 |
| 2 | nvtx_kern_sum / nvtx_gpu_proj_sum | 每阶段构成（免 SQL） |
| 3 | GUI | 时间线肉眼校验：对齐、稳定、无意外拷贝 |
| 4 | sqlite 自查 | 任意自定义问题：构成分类、步间一致性、窗口质量 |
| → | ncu | 对头号 kernel 追问"为什么慢" |

跑完第一步把两个 CSV 的 Instances/百分比贴给我（或跑完 2/4 步把 `nvtx_gpu_proj_sum`、backward 构成表贴出来），我可以帮你按第 11 章的预测逐项核对哪些符合、哪些是异常信号。


先把归为 `other` 的最大几项看一遍，补全规则。

**GPU 利用率自检**：`kernel 总时间 ÷ 步的实际时间`（来自 `cuda_gpu_kern_sum` 与 `nvtx_sum` 的 `step`）。

| 比值 | 含义 |
|---|---|
| 接近 1 | GPU 受限，优化 kernel 本身（这时才值得上 ncu） |
| 明显低于 1（如小于 0.6） | **发射受限**，GPU 在等 CPU，时间线上 kernel 之间有明显空隙；ncu 帮不上忙，要减少 kernel 数（融合、`torch.compile`、CUDA graph） |

**时间线（Nsight Systems GUI）**：把 `.nsys-rep` 下载到本地，GUI 版本不低于生成它的 nsys（`nsys --version` 对照）。

| 行 | 看什么 |
|---|---|
| NVTX | 区间的层级与宽度，先定位 `forward`、`backward` |
| CUDA HW → Kernels | kernel 是否首尾相接，空隙意味着发射受限 |
| 主线程 → CUDA API | kernel 对应的发射调用，点击 kernel 可跳转 |
| 另一个线程 | autograd 的反向线程（12.8） |

操作：先缩放到一个 `block`，数 kernel 个数（对照 11.5(4) 的约 50），再看哪几个最宽。

**产出"待深挖清单"**：从 `cuda_gpu_kern_sum` 里挑 3 到 6 个 kernel（总时间占比最大的 GEMM、softmax、最大的逐元素、reduce），记下完整名字，供 12.10 过滤。


## 12.8 backward 与 optimizer 的归因

forward 与 optimizer 都在主线程发射，NVTX 投影正常。

**backward 的问题**：PyTorch 的反向在 autograd 引擎的**工作线程**上发射 kernel，而 `backward` 区间在主线程里打（主线程只是阻塞等待）。nsys 的 NVTX 投影按发射线程匹配区间，所以**很可能**出现 `backward` 区间投影到的 kernel 很少或为 0。这是基于机制的判断，不是对某个版本的实测，请先验证：

1. 看 `nvtx_gpu_proj_sum` 或 `nvtx_kern_sum` 里 `backward` 的 kernel 数。若远少于 forward 的约 2 倍，就是这个问题。
2. 备选方案：

| 方案 | 做法 | 说明 |
|---|---|---|
| A. 时间窗归因 | 取 `backward` 区间的起止时间，从 `cuda_gpu_trace` 里选起始时间落在窗内的 kernel | 配合 12.5 每阶段末尾同步，窗内的 GPU 工作就是该阶段的。最稳妥 |
| B. `emit_nvtx` | `with torch.autograd.profiler.emit_nvtx():` 包住整个 profile 区间 | 为每个 autograd 算子（含反向）自动打 NVTX，名字形如 `MmBackward0`，反向区间在引擎线程上，投影正确 |
| C. 新版 nsys 的 PyTorch 选项 | `nsys profile --help` 里查 `--pytorch` | 不确定你的版本是否支持 |

作业里"forward 与 forward+backward 中同一个 kernel 占比"这类问题，方案 A 就够。



## 12.9 把 kernel 对应回代码

| 方法 | 做法 | 可靠度 |
|---|---|---|
| A. 单算子隔离 | 写一个小循环只跑一个算子（`torch.softmax`、一次 `einsum`），用 nsys 看它产生的 kernel 名 | **最可靠**，对 6 到 8 个关键算子各做一次，建立名字字典 |
| B. 细粒度 NVTX | 12.4 的区间，配合 `nvtx_kern_sum` | 可靠，缺点是 backward 归因问题 |
| C. `torch.profiler` | `aten::` 级别的 CUDA 时间，更接近代码（12.11） | 与 nsys 互补 |
| D. GUI 点击 | 点 kernel，跳到发射它的 CUDA API 调用与线程 | 适合个别 kernel |

**kernel 名速查**（名字依库版本与架构（sm_120）而变，**以实测为准**）：

| 类别 | 常见名字特征 | 对应代码 |
|---|---|---|
| GEMM | `gemm`、`xmma`、`cutlass`、`cublas`、`sgemm` | `Linear`、$QK^\top$、$PV$、`lm_head` |
| 逐元素 | `elementwise`、`vectorized_elementwise_kernel` | RMSNorm 的乘除、RoPE、缩放、`where`、SiLU、残差 |
| 归约 | `reduce_kernel` | RMSNorm 的 `mean` |
| softmax | `SoftMax` | `softmax` 或其内部实现 |
| 拷贝 / cat | `copy`、`CatArrayBatchedCopy` | `.contiguous()`、`concat`、dtype 转换 |
| 索引 | `index`、`embedding`、`scatter` | embedding 的查表与反向 |
| 优化器 | `multi_tensor_apply`，或一大批小 kernel | AdamW |

staff 的 AdamW 若是逐参数 Python 循环，nsys 里会有数量极多的小 kernel，`cuda_api_sum` 中发射开销占主导。这正是 11.5(1) 里 optimizer 实测会远高于带宽下限的原因。



## 12.10 用 ncu 验证"为什么"

nsys 告诉你**哪个 kernel 占时间**，11 章里"注意力是访存密集""大 GEMM 接近峰值"这类结论，用时间占比只能**推断**。ncu 能**直接验证**：它看单个 kernel 内部，给出算力与带宽的利用率、占用率、访存效率。

### 什么时候用，什么时候不用

| 情况 | 用什么 |
|---|---|
| GPU 利用率低、kernel 之间有空隙（发射受限） | nsys 的时间线，ncu 帮不上忙 |
| 某个 kernel 占时间多，想知道它离硬件极限有多远 | **ncu** |
| 想确认某个 kernel 是算力受限还是带宽受限 | **ncu** |
| 想比较融合或优化前后同一个 kernel | **ncu**（比较指标，不比较 duration） |

### 环境与权限

- 先确认版本与支持：`ncu --version`、`ncu --list-sections`。你的卡是 sm_120（Blackwell 消费级），需要较新的 ncu，**这一点是按一般规律判断，请在本机核实**。
- 读取 GPU 性能计数器可能需要权限，共享服务器上常见报错 `ERR_NVGPUCTRPERM`，需要管理员放开，或用有权限的账号运行。
- 指标名随版本和架构变化，用 `ncu --query-metrics` 或直接用 section，不要照抄旧教程里的指标名。

### 只抓你关心的 kernel

ncu 会把每个被抓取的 kernel **重放多次**，不加限制会慢到跑不完。三种过滤方式可以组合：

| 方式 | 选项 | 说明 |
|---|---|---|
| 按名字 | `-k regex:softmax` | 名字来自 12.7 的待深挖清单 |
| 按次数 | `-s 20 -c 3` | 跳过前 20 次匹配，只抓接下来的 3 次 |
| 按 NVTX 区间 | `--nvtx --nvtx-include "attn.softmax/"` | 复用 12.4 的标注，需要 `--nvtx-fine` |

由于 12.5 的脚本用了 `profiler.start()`，让 ncu 同样只在这之后抓取：

```bash
mkdir -p results/ncu
CUDA_VISIBLE_DEVICES=0 uv run ncu -o results/ncu/softmax --force-overwrite --profile-from-start off -k regex:softmax -c 3 --set full python scripts/profile_run.py --model small --ctx 512 --batch 4 --mode fwd --warmup 3 --steps 1 --nvtx-fine
```

| 选项 | 作用 |
|---|---|
| `--profile-from-start off` | 配合 `cudaProfilerStart`，预热不被抓 |
| `--set full` | 全套指标（慢）；先快速看一眼用 `--set basic` |
| `--section <名字>` | 只采指定 section，更快 |
| `--clock-control none` | 默认会锁定时钟，使结果稳定；想看接近真实运行的频率时关掉 |

先用 `-c 1` 试通流程，再加量。

### 先看哪几个 section

| 想回答 | 看什么 |
|---|---|
| 算力还是带宽受限 | `Speed Of Light`：Compute (SM) Throughput 与 Memory Throughput 哪个更接近 100% |
| 在 roofline 上的位置 | roofline 图（`SpeedOfLight_RooflineChart`），看点落在斜线区还是平线区 |
| 访存是否高效 | `Memory Workload Analysis`：DRAM 吞吐、L2 命中率、访存合并情况 |
| 并行度够不够 | `Launch Statistics`（grid 与 block 大小）、`Occupancy`（理论与实际占用率、寄存器、共享内存） |
| 线程停在哪 | `Warp State Statistics`（stall 原因：等访存、等同步、等指令） |

命令行打印文本摘要：`ncu --import results/ncu/softmax.ncu-rep --page details`。GUI 里把 `.ncu-rep` 下载到本地打开，版本不低于生成它的 ncu。


```bash
mkdir -p results/ncu
sudo env CUDA_VISIBLE_DEVICES=0 /usr/local/cuda/bin/ncu \
  -o results/ncu/block0 -f \
  --profile-from-start off \
  --nvtx --nvtx-include "TransformerBlock/" \
  -c 52 --set basic \
  /data2/c220/hz/cs336/.venv/bin/python scripts/profile_run.py \
    --model small --ctx 512 --batch 4 --mode fwd --warmup 2 --steps 1 --nvtx-fine

```

In [7]:
import pandas as pd, re
pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 70)

d = pd.read_csv("../results/ncu/block0.csv")
d = d[pd.to_numeric(d["ID"], errors="coerce").notna()].copy()    # 去掉单位行
print("抓到的 kernel 数:", len(d))                                # 应为 52

def short(n, w=60):
    n = re.sub(r"^void\s+", "", str(n)); n = re.sub(r"<.*", "<…>", n)
    return n[:w]
def num(s): return pd.to_numeric(s.astype(str).str.replace(",", ""), errors="coerce")
def col(sub): return next(c for c in d.columns if sub in c)

t = pd.DataFrame({
    "kernel":  d["Kernel Name"].map(short),
    "grid":    d[col("launch__grid_size")],
    "compute%": num(d[col("sm__throughput.avg.pct_of_peak_sustained_elapsed")]),
    "memory%":  num(d[col("gpu__compute_memory_throughput.avg.pct_of_peak_sustained_elapsed")]),
    "dram%":    num(d[col("gpu__dram_throughput.avg.pct_of_peak_sustained_elapsed")]),
}).round(1)
print(t.to_string())

抓到的 kernel 数: 52
                                  kernel    grid  compute%  memory%  dram%
1   at::vectorized_elementwise_kernel<…>   1,536       3.6     67.8   67.8
2                   at::reduce_kernel<…>     128       6.6     63.3   63.3
3   at::vectorized_elementwise_kernel<…>       2       0.0      6.1    0.7
4   at::vectorized_elementwise_kernel<…>       2       0.0      5.9    0.8
5              at::elementwise_kernel<…>   6,144      33.9     60.5   60.5
6              at::elementwise_kernel<…>   6,144      35.8     63.4   63.4
7                magma_sgemmEx_kernel<…>     384      59.7     76.1    6.1
8                magma_sgemmEx_kernel<…>     384      59.9     76.3    6.3
9                magma_sgemmEx_kernel<…>     384      59.7     76.1    6.1
10             at::elementwise_kernel<…>   3,072      28.2     67.6   67.6
11             at::elementwise_kernel<…>   3,072      29.0     68.3   68.3
12  at::vectorized_elementwise_kernel<…>     768       2.4     67.6   67.6
13      

In [8]:
# 先确认那几个关键 kernel 的名字
for i in [7, 26, 36, 46, 51]:          # 表里的行号从 1 开始
    print(i, re.sub(r"^void\s+", "", d.iloc[i-1]["Kernel Name"])[:130])

7 magma_sgemmEx_kernel<float, float, float, 1, 0, 6, 4, 6, 3, 4>(int, int, int, BatchedTensor, int, BatchedTensor, int, BatchedTenso
26 magma_sgemmEx_kernel<float, float, float, 1, 0, 6, 4, 6, 3, 4>(int, int, int, BatchedTensor, int, BatchedTensor, int, BatchedTenso
36 cutlass::Kernel2<cutlass_80_simt_sgemm_64x64_8x5_nn_align1>(T1::Params)
46 magma_sgemmEx_kernel<float, float, float, 1, 0, 6, 4, 6, 3, 4>(int, int, int, BatchedTensor, int, BatchedTensor, int, BatchedTenso
51 cutlass::Kernel2<cutlass_80_simt_sgemm_128x256_8x4_tn_align1>(T1::Params)


In [3]:
# # a) softmax 的 5 个（forward，NVTX 过滤可用）
# ncu -o results/ncu/softmax -f --profile-from-start off \
#   --nvtx --nvtx-include "attn.softmax/" -c 5 --set full \
#   python scripts/profile_run.py --model small --ctx 512 --batch 4 --mode fwd --warmup 2 --steps 1 --nvtx-fine

# # b) 对照：同样的 softmax，不清缓存
# ... -o results/ncu/softmax_warm --cache-control none ...

# # c) 三个代表性 GEMM（在第一个 block 内按顺序取，-s 是跳过的个数）
# ... --nvtx --nvtx-include "TransformerBlock/" -s 6  -c 1 -o results/ncu/gemm_qkv ...   # 7 号
# ... --nvtx --nvtx-include "TransformerBlock/" -s 25 -c 1 -o results/ncu/gemm_qk  ...   # 26 号
# ... --nvtx --nvtx-include "TransformerBlock/" -s 50 -c 1 -o results/ncu/gemm_w2  ...   # 51 号

import os, getpass
os.environ['SUDO_PASS'] = getpass.getpass('sudo password: ')
os.makedirs('results/ncu', exist_ok=True)
print('ready')


ready


In [5]:
%%bash
set -e
sudo -S -p '' env CUDA_VISIBLE_DEVICES=0 /usr/local/cuda/bin/ncu \
  -o ../results/ncu/softmax -f \
  --profile-from-start off \
  --nvtx --nvtx-include "attn.softmax/" \
  -c 5 --set full \
  /data2/c220/hz/cs336/.venv/bin/python ../scripts/profile_run.py \
    --model small --ctx 512 --batch 4 --mode fwd --warmup 2 --steps 1 --nvtx-fine \
  <<< "$SUDO_PASS"
sudo chown c220:c220 ../results/ncu/softmax.ncu-rep


==PROF== Connected to process 2038716 (/home/c220/.local/share/uv/python/cpython-3.13.17-linux-x86_64-gnu/bin/python3.13)
==PROF== Profiling "reduce_kernel" - 0 (1/5): 0%....50%....100% - 40 passes
==PROF== Profiling "elementwise_kernel" - 1 (2/5): 0%....50%....100% - 40 passes
==PROF== Profiling "vectorized_elementwise_kernel" - 2 (3/5): 0%....50%....100% - 40 passes
==PROF== Profiling "reduce_kernel" - 3 (4/5): 0%....50%....100% - 40 passes
==PROF== Profiling "elementwise_kernel" - 4 (5/5): 0%....50%....100% - 40 passes
device: NVIDIA GeForce RTX 5070 Ti  torch 2.11.0+cu130
model=small params=128.63M optimizer=staff AdamW mode=fwd ctx=512 batch=4 dtype=float32
warmup=2 steps=1 nvtx_fine=True emit_nvtx=False (TF32 off)
[提醒] 每 phase 同步 + profiler 开销：只看构成，绝对时间以 benchmark.py 为准
[自检] 每步各区间次数（实测 / 预期）：
  RMSNorm                            25.0 / 25
  Linear                             85.0 / 85
  CausalMultiHeadSelfAttention       12.0 / 12
  SwiGLU                             12.0 / 12
  

In [7]:
%%bash
set -e
sudo -S -p '' env CUDA_VISIBLE_DEVICES=0 /usr/local/cuda/bin/ncu \
  -o ../results/ncu/softmax_warm -f \
  --profile-from-start off \
  --nvtx --nvtx-include "attn.softmax/" \
  -c 5 --set full \
  --cache-control none \
  /data2/c220/hz/cs336/.venv/bin/python ../scripts/profile_run.py \
    --model small --ctx 512 --batch 4 --mode fwd --warmup 2 --steps 1 --nvtx-fine \
  <<< "$SUDO_PASS"
sudo chown c220:c220 ../results/ncu/softmax_warm.ncu-rep


==WARNING== Note: Running with uncontrolled GPU caches. Profiling results may be inconsistent.
==PROF== Connected to process 2039582 (/home/c220/.local/share/uv/python/cpython-3.13.17-linux-x86_64-gnu/bin/python3.13)
==PROF== Profiling "reduce_kernel" - 0 (1/5): 0%....50%....100% - 40 passes
==PROF== Profiling "elementwise_kernel" - 1 (2/5): 0%....50%....100% - 40 passes
==PROF== Profiling "vectorized_elementwise_kernel" - 2 (3/5): 0%....50%....100% - 40 passes
==PROF== Profiling "reduce_kernel" - 3 (4/5): 0%....50%....100% - 40 passes
==PROF== Profiling "elementwise_kernel" - 4 (5/5): 0%....50%....100% - 40 passes
device: NVIDIA GeForce RTX 5070 Ti  torch 2.11.0+cu130
model=small params=128.63M optimizer=staff AdamW mode=fwd ctx=512 batch=4 dtype=float32
warmup=2 steps=1 nvtx_fine=True emit_nvtx=False (TF32 off)
[提醒] 每 phase 同步 + profiler 开销：只看构成，绝对时间以 benchmark.py 为准
[自检] 每步各区间次数（实测 / 预期）：
  RMSNorm                            25.0 / 25
  Linear                             85.0 / 85
 

In [8]:
%%bash
set -e
NCU=/usr/local/cuda/bin/ncu
PY=/data2/c220/hz/cs336/.venv/bin/python
COMMON="--profile-from-start off --nvtx --nvtx-include TransformerBlock/ --set basic"
ARGS="--model small --ctx 512 --batch 4 --mode fwd --warmup 2 --steps 1 --nvtx-fine"

sudo -S -p '' env CUDA_VISIBLE_DEVICES=0 $NCU -o ../results/ncu/gemm_qkv -f $COMMON -s 6  -c 1 \
  $PY ../scripts/profile_run.py $ARGS <<< "$SUDO_PASS"

sudo -S -p '' env CUDA_VISIBLE_DEVICES=0 $NCU -o ../results/ncu/gemm_qk  -f $COMMON -s 25 -c 1 \
  $PY ../scripts/profile_run.py $ARGS <<< "$SUDO_PASS"

sudo -S -p '' env CUDA_VISIBLE_DEVICES=0 $NCU -o ../results/ncu/gemm_w2  -f $COMMON -s 50 -c 1 \
  $PY ../scripts/profile_run.py $ARGS <<< "$SUDO_PASS"

sudo chown c220:c220 ../results/ncu/gemm_*.ncu-rep


==PROF== Connected to process 2040301 (/home/c220/.local/share/uv/python/cpython-3.13.17-linux-x86_64-gnu/bin/python3.13)
==PROF== Profiling "magma_sgemmEx_kernel" - 0 (1/1): 0%....50%....100% - 9 passes
device: NVIDIA GeForce RTX 5070 Ti  torch 2.11.0+cu130
model=small params=128.63M optimizer=staff AdamW mode=fwd ctx=512 batch=4 dtype=float32
warmup=2 steps=1 nvtx_fine=True emit_nvtx=False (TF32 off)
[提醒] 每 phase 同步 + profiler 开销：只看构成，绝对时间以 benchmark.py 为准
[自检] 每步各区间次数（实测 / 预期）：
  RMSNorm                            25.0 / 25
  Linear                             85.0 / 85
  CausalMultiHeadSelfAttention       12.0 / 12
  SwiGLU                             12.0 / 12
  TransformerBlock                   12.0 / 12
  Embedding                           1.0 / 1
  sdpa                               12.0 / 12
done: 1 steps profiled (capture range closed)
==PROF== Disconnected from process 2040301
==PROF== Report: /data2/c220/hz/cs336/documents/../results/ncu/gemm_qkv.ncu-rep
==PROF== Connecte

> `ncu --import results/ncu/softmax.ncu-rep --page details`

In [10]:
%%bash
ncu --import ../results/ncu/gemm_qkv.ncu-rep --page details | head -80


[2040301] python3.13@127.0.0.1
  void magma_sgemmEx_kernel<float, float, float, 1, 0, 6, 4, 6, 3, 4>(int, int, int, BatchedTensor, int, BatchedTensor, int, BatchedTensor, int, BatchedTensor, int, int, int, const T1 *, const T1 *, T1, T1, int, cublasLtEpilogue_t, int, const void *, long) (12, 32, 1)x(8, 16, 1), Context 1, Stream 7, Device 0, CC 12.0

    NVTX Push/Pop Stack for Thread 2040301:
      <default domain>
        <0,step>
        <1,forward>
        <2,TransformerBlock>
        <3,CausalMultiHeadSelfAttention>
        <4,Linear>
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz        13.78
    SM Frequency                    Ghz         1.96
    Elapsed Cycles                cycle      316,505
    Memory Throughput                 %        76.31
    DRAM Throughput                   %  

结论先说：这是一个卡在 L1 缓存上的小 GEMM，不是 compute-bound，也不是 DRAM-bound——ncu 自己已经在 OPT 提示里点明了"L1 bottleneck"。

In [11]:
%%bash
ncu --import ../results/ncu/softmax_warm.ncu-rep --page details | head -80

[2039582] python3.13@127.0.0.1
  void reduce_kernel<512, 1, ReduceOp<float, MaxOps<float>, unsigned int, float, 4, 4>>(T3) (1536, 1, 1)x(32, 16, 1), Context 1, Stream 7, Device 0, CC 12.0

    NVTX Push/Pop Stack for Thread 2039582:
      <default domain>
        <0,step>
        <1,forward>
        <2,TransformerBlock>
        <3,CausalMultiHeadSelfAttention>
        <4,attn.softmax>
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz        13.78
    SM Frequency                    Ghz         2.27
    Elapsed Cycles                cycle      156,170
    Memory Throughput                 %        83.30
    DRAM Throughput                   %        83.30
    Duration                         us        68.61
    L1/TEX Cache Throughput           %        19.45
    L2 Cache Throughput               %

In [12]:
%%bash
ncu --import ../results/ncu/gemm_w2.ncu-rep --page details | head -80

[2040569] python3.13@127.0.0.1
  void Kernel2<cutlass_80_simt_sgemm_128x256_8x4_tn_align1>(Params) (64, 1, 7)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 12.0

    NVTX Push/Pop Stack for Thread 2040569:
      <default domain>
        <0,step>
        <1,forward>
        <2,TransformerBlock>
        <3,SwiGLU>
        <4,Linear>
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz        13.79
    SM Frequency                    Ghz         2.27
    Elapsed Cycles                cycle    1,204,079
    Memory Throughput                 %        62.84
    DRAM Throughput                   %         7.62
    Duration                         us       529.95
    L1/TEX Cache Throughput           %        67.20
    L2 Cache Throughput               %        32.67
    SM Active Cycles              cycle 

- 要知道两个 GEMM 具体卡在哪，还需要被截断的三块：Compute Workload Analysis（FMA 利用率）、Warp State Statistics（stall 原因）、Memory Workload Analysis（共享内存）

In [19]:
%%bash
set -e
NCU=/usr/local/cuda/bin/ncu
PY=/data2/c220/hz/cs336/.venv/bin/python
COMMON="--profile-from-start off --nvtx --nvtx-include TransformerBlock/ --set full"
ARGS="--model small --ctx 512 --batch 4 --mode fwd --warmup 2 --steps 1 --nvtx-fine"

sudo -S -p '' env CUDA_VISIBLE_DEVICES=0 $NCU -o ../results/ncu/gemm_qkv_full -f $COMMON -s 6  -c 1 \
  $PY ../scripts/profile_run.py $ARGS <<< "$SUDO_PASS"

sudo -S -p '' env CUDA_VISIBLE_DEVICES=0 $NCU -o ../results/ncu/gemm_qk_full  -f $COMMON -s 25 -c 1 \
  $PY ../scripts/profile_run.py $ARGS <<< "$SUDO_PASS"

sudo -S -p '' env CUDA_VISIBLE_DEVICES=0 $NCU -o ../results/ncu/gemm_w2_full  -f $COMMON -s 50 -c 1 \
  $PY ../scripts/profile_run.py $ARGS <<< "$SUDO_PASS"

sudo chown c220:c220 ../results/ncu/gemm_*.ncu-rep

==PROF== Connected to process 2069279 (/home/c220/.local/share/uv/python/cpython-3.13.17-linux-x86_64-gnu/bin/python3.13)
==PROF== Profiling "magma_sgemmEx_kernel" - 0 (1/1): 0%....50%....100% - 40 passes
device: NVIDIA GeForce RTX 5070 Ti  torch 2.11.0+cu130
model=small params=128.63M optimizer=staff AdamW mode=fwd ctx=512 batch=4 dtype=float32
warmup=2 steps=1 nvtx_fine=True emit_nvtx=False (TF32 off)
[提醒] 每 phase 同步 + profiler 开销：只看构成，绝对时间以 benchmark.py 为准
[自检] 每步各区间次数（实测 / 预期）：
  RMSNorm                            25.0 / 25
  Linear                             85.0 / 85
  CausalMultiHeadSelfAttention       12.0 / 12
  SwiGLU                             12.0 / 12
  TransformerBlock                   12.0 / 12
  Embedding                           1.0 / 1
  sdpa                               12.0 / 12
done: 1 steps profiled (capture range closed)
==PROF== Disconnected from process 2069279
==PROF== Report: /data2/c220/hz/cs336/documents/../results/ncu/gemm_qkv_full.ncu-rep
==PROF== Co

In [20]:
%%bash
for f in gemm_qkv_full gemm_w2_full; do
  echo "=== $f"
  ncu --import ../results/ncu/$f.ncu-rep --page details \
   | awk '/^ *Section: /{p = ($0 ~ /Compute Workload|Scheduler|Warp State|Memory Workload|Instruction Stat/)} p'
done

=== gemm_qkv_full
    Section: Compute Workload Analysis
    -------------------- ----------- ------------
    Metric Name          Metric Unit Metric Value
    -------------------- ----------- ------------
    Executed Ipc Active   inst/cycle         2.65
    Executed Ipc Elapsed  inst/cycle         2.39
    Issue Slots Busy               %        66.19
    Issued Ipc Active     inst/cycle         2.65
    SM Busy                        %        66.19
    -------------------- ----------- ------------

    INF   FMA is the highest-utilized pipeline (50.0%) based on active cycles, taking into account the rates of its     
          different instructions. It executes 32-bit floating point (FADD, FMUL, FMAD, ...) and integer (IMUL, IMAD)    
          operations. It is well-utilized, but should not be a bottleneck.                                              

    Section: Memory Workload Analysis
    -------------------- ----------- ------------
    Metric Name          Metric Unit Met


### 解读规则

一个粗略的判据，用 Speed Of Light 的两个百分比：

| Compute (SM) | Memory | 结论 |
|---|---|---|
| 高 | 低 | 算力受限，接近峰值就没有大空间了 |
| 低 | 高 | **带宽受限**，要减少访存（融合、降精度）而不是算得更快 |
| 都低 | 都低 | 并行度不足或 kernel 太小：看 grid 大小、occupancy，通常是延迟受限 |
| 都高 | 都高 | 已经接近硬件极限 |

### 对 11 章预测的直接验证

| kernel | 预测（11.2、11.4） | ncu 应看到 |
|---|---|---|
| 缩放、mask、softmax、SiLU、残差等逐元素 | 访存受限 | Memory Throughput 高（接近 5070 Ti 的 769 GB/s 量级），Compute 很低 |
| $QK^\top$、$PV$ 的 GEMM | 计算为主但强度偏低 | Compute 利用率达不到大 GEMM 的水平，看 occupancy 与形状 |
| FFN 的大 GEMM | 计算受限，接近峰值 | Compute Throughput 高，对应 11.5 里 $P=33.4$ TFLOP/s |
| RMSNorm 的 reduce | 访存受限，且可能并行度偏低 | Memory 偏高或两者都低，看 occupancy |
| 逐参数 AdamW 的小 kernel | 发射受限 | duration 极短、occupancy 低；这类问题用 nsys 的空隙分析，ncu 意义有限 |

### 注意

| 坑 | 说明 |
|---|---|
| **duration 不可信** | 重放、锁频使 ncu 报告的 kernel 时间不能与 nsys 或 benchmark 直接比，只用指标与比例 |
| 一次只抓少数几个 | 全程序重放会极慢，先 `-c 1` |
| 形状要与真实运行一致 | 想验证 small、$T=512$ 的结论，就用同样的配置跑，不要换成缩小的玩具形状 |
| fp32 与 TF32 | 确认 TF32 关闭（第 10 章的设置），否则 GEMM 走的是不同的路径，与预测不符 |
| 指标名与版本 | sm_120 上老教程里的指标名可能失效，以 `--query-metrics` 与 GUI 为准 |


**benchmark 的结果出来后，预测和实测大体吻合，但吻合有一部分是误差相互抵消的结果。** forward 总时间的预测很准（1.02×），可是 GEMM 比预测慢 1.5 倍，访存类比预测快 35%，两者刚好抵消。

## 1. 三个工具互相印证

benchmark 只给阶段总时间，后两行用中位数相减得到。

| 阶段 | benchmark | nsys kernel 耗时之和 | 比值 | 占整步 |
|---|---|---|---|---|
| forward | 37.72 ms | 37.37 ms | 0.99 | 29.5% |
| backward | 77.18 ms | 73.85 ms | 0.96 | 60.4% |
| optimizer | 12.95 ms | 10.40 ms | 0.80 | 10.1% |
| 整步 | 127.85 ms | 121.6 ms | 0.95 | 100% |

- forward 和 backward 的墙钟时间约等于 kernel 耗时之和，说明这两段受 GPU 限制，nsys 的构成可信。
- nsys 里 optimizer 的 GPU 利用率是 0.49，而 benchmark 里约为 0.80。原因是 nsys 把 CPU 发射开销放大了。这里假设没有 profiler 时 kernel 耗时不变。
- nsys 的整步窗口是 137.0 ms，比 benchmark 的 127.85 ms 高约 7%，来自同步和采集开销。所以 nsys 只看比例，绝对时间用 benchmark。
- 显存：forward 0.71 GiB，forward+backward 4.06 GiB，完整步 5.04 GiB。其中 Adam 状态占 0.98 GiB，与 8 B/参数 × 128.6M 的估算一致。

## 2. 11.5 回填表

判定标准是实测÷预测在 0.8～1.25 之间算成立。

**(1) 阶段时间**

| 项 | 预测 | 实测 | 实测÷预测 | 结论 |
|---|---|---|---|---|
| forward | 约 37 ms | 37.72 ms | 1.02 | 成立 |
| backward | 约 72 ms | 77.18 ms | 1.07 | 成立 |
| optimizer | 下限 4.7 ms，"明显高于下限" | 12.95 ms | 下限的 2.76 倍 | 定性成立 |

backward 的预测在 11.5 里是"待估"，这个 72 ms 是我按"理想 32.0 ms + 2 × 访存 20 ms"补算的。

**(2) 时间占比**（forward 的 kernel 耗时）

| 项 | 预测 | 实测 | 比值 |
|---|---|---|---|
| 注意力（scores + softmax + av） | 约 35% | 29.4% | 0.84 |
| FFN（SwiGLU） | 约 40% | 44.7% | 1.12 |
| 其余 | 约 25% | 25.9% | 1.04 |

三项都成立，11.4 的核心结论也得到验证：注意力的 FLOPs 只占 7.2%，却占一个 block 时间的 31.4%（10.99 ÷ 35.03 ms）。其中 QKᵀ 与 PV 两个 GEMM 约 2.4 ms，占 forward 的 6.4%，与 FLOPs 占比一致，其余约 8.6 ms 全是访存类运算。

**(3) 按类别（forward）**

| 类别 | kernel 数：预测→实测 | 时间占比：预测→实测 |
|---|---|---|
| gemm | 109 → 109，比值 1.00 | 45% → 65.1%，比值 1.45，**偏离** |
| elementwise | 约 300 → 456，比值 1.52，**偏离** | 40% → 30.3%，比值 0.76，**偏离** |
| reduce | 约 25 → 52，比值 2.08，**偏离** | 3% → 3.8%，比值 1.27，边界 |
| copy / cat | 约 40 → 24，比值 0.60，**偏离** | 3% → 0.8% |
| softmax（其中） | 12 → 60 | 5% → 14.6%，约 2.9 倍，**偏离** |

softmax 的 60 个 kernel 里有 36 个 elementwise 和 24 个 reduce，已经算在上面两行里。copy 的 24 个可能是 RoPE 里每层 2 个 cat，这是我的推测。

**(4) 按区间（forward）**

| 项 | 预测 | 实测 | 比值 |
|---|---|---|---|
| block × 12，kernel 数 | 约 600 | 624 | 1.04 |
| block × 12，时间 | 34.8 ms | 35.03 ms | 1.01 |
| block 之外（embed + ln_final + lm_head + loss） | 约 2.15 ms | 2.34 ms | 1.09 |

**(5) 显存**

| 项 | 预测 | 实测 | 比值 |
|---|---|---|---|
| 峰值 | 6.16 GiB | 5.04 GiB | 0.82，边界 |
| 其中激活部分 | 4.24 GiB | 约 3.12 GiB | 0.74，偏低 |

预测值是我用 11.6 的函数算的，取 d=768、L=12、h=12、B=4、T=512，c=2，other=20，P=128.6M。V ≈ 10000 是我从 lm_head 的 FLOPs 反推的，请核对。

## 3. 最出乎意料的三项

**① forward 总时间准，是两个分量方向相反的误差抵消了。**

| 分量 | 预测 | 实测 | 比值 |
|---|---|---|---|
| GEMM | 16.0 ms | 24.33 ms | 1.52 |
| 访存类 | 约 20 ms | 13.04 ms | 0.65 |

- **GEMM 偏慢的证据**：forward 的 GEMM 只有 21.9 TFLOP/s，约为 P = 33.4 的 66%。ncu 看到两个代表性 GEMM 的 FMA 利用率都约 50%（qkv 50.0%，w2 48.8%），原因不同：qkv 的 FMA 只占指令的 71%，w2 每个调度器只有 2 个 warp、44.7% 的周期无指令可发。TF32 关闭，走的是 `cutlass_80_simt_sgemm`，没有用 Tensor Core。
- **访存类偏快**：ncu 里大张量逐元素的 DRAM 约 85～87%，实测峰值带宽约 882 GB/s，高于 11.5 用的 769 GB/s，只能解释约 13%。其余可能是我预测的读写遍数偏多，这一点是假设，没有验证。
- **backward 的 GEMM 反而更高效**：27.3 TFLOP/s，耗时是 forward 的 1.61 倍而非 2 倍。forward 的 85 个 Linear 里 73 个用 `magma_sgemmEx`，backward 用的是更大 tile 的 cutlass，这个对应关系我只在 kernel 名上看到，没有 ncu 证据。
- **下一步**：改用 TF32 或 BF16 走 Tensor Core，并重新校准 P。

**② softmax：预测 5%，实测 14.6%，且是 5 个 kernel。**

- 每次 softmax 是 max、sub、exp、sum、div 五个 kernel，每层搬运 8 遍 T² 张量，合计 453 µs。
- ncu 看到其中的 max reduce 的 DRAM 是 83.3%（734.6 GB/s），且搬运量 50 MB 与形状对得上。其余 4 个用 nsys 时间反推，合计约 889 GB/s，约等于 DRAM 峰值。
- 融合版只需约 2 遍，理论上约 4 倍。按这个估算，约 4 ms/步，占整步约 3%。在 ctx 512 下收益有限，随 T² 增长才明显。

**③ optimizer：预测的是带宽下限，实测是 2.76 倍。**

- 1776 个 kernel，平均每个只有 5.9 µs；如果 1776 = 111 × 16，就是逐参数循环，每个参数 16 个 kernel，这取决于你的参数张量个数。
- 即使只算 GPU 上的时间，10.4 ms 也已是带宽下限的 2.2 倍，折算有效带宽约 346 GB/s，不到峰值的一半。
- 下一步：用 foreach 或融合版 AdamW，上限约 8 ms/步，占整步约 6%。

## 4. 没有验证、需要保留的限定

- 显存激活偏低 26%，原因还没有定位。候选：c 或 other_per_layer 取大了；梯度在 backward 才分配，峰值时刻 16P 不一定同时存在。验证用 `saved_tensors_hooks` 或 12.11 的内存快照。
- 上面按类别、按区间的时间用的是 nsys 的 kernel 耗时之和。forward 阶段与 benchmark 的比值是 0.99，可信；backward 的类别拆分来自差分法，账是平的。
- QKᵀ、PV、w2 与具体 kernel 名的对应，是我用区间总量推出来的。
- ncu 的 duration 与 nsys 对不上（softmax max：68.6 µs 对 51.5 µs），只用了利用率和比例。
- 不同 kernel 的 SM 频率是 1.96 或 2.27 GHz，Compute% 不宜横向比较。




## 12.11 配套工具：torch.profiler 与显存

**`torch.profiler`**（算子级视角）：

```python
from torch.profiler import profile, ProfilerActivity
with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], record_shapes=True) as prof:
    train_step(); torch.cuda.synchronize()
print(prof.key_averages(group_by_input_shape=True).table(sort_by="cuda_time_total", row_limit=20))
prof.export_chrome_trace("results/trace.json")      # 可在 Perfetto 或 chrome://tracing 打开
```

新版本里排序键可能叫 `device_time_total`，按报错提示改。它给出 `aten::mm`、`aten::bmm`、`aten::softmax` 等的 CUDA 时间，适合回答"每类 PyTorch 算子占多少"。

**显存峰值**（对照 11.6）：

```python
torch.cuda.reset_peak_memory_stats()
train_step(); torch.cuda.synchronize()
print(torch.cuda.max_memory_allocated() / 2**30, est_peak_gib(...))   # 实测 vs 预估
```

实测与预估的差值就是要在注解里解释的内容（份数估错、临时 buffer、allocator 碎片）。`max_memory_reserved` 与 `max_memory_allocated` 之差反映碎片。按组成验证用第 8 章的 `saved_tensors_hooks`，统计各层保存的张量。

**显存快照**（对应 3.8）：

```python
torch.cuda.memory._record_memory_history(max_entries=100000)
train_step(); torch.cuda.synchronize()
torch.cuda.memory._dump_snapshot("results/mem_small_512.pickle")
torch.cuda.memory._record_memory_history(enabled=None)
```

把 pickle 下载到本地，拖到 pytorch.org/memory_viz（浏览器本地解析）。读法：

- "Active Memory Timeline"：峰值出现在哪个阶段（预期是 forward 末尾或 backward 开始）；
- 悬停色块看分配时的调用栈，识别 $T^2$ 大小的分数矩阵、$N\times d_{ff}$ 的 FFN 中间量。

**分布式（Part 3 预告）**：多进程可用 `%q{RANK}` 在输出文件名里区分，每个 rank 各自调用 `profiler.start/stop`，到 Part 3 再展开。



## 12.12 常见坑

| 坑 | 后果 | 处理 |
|---|---|---|
| 没有预热，或预热被抓进文件 | 看到 cuBLAS 初始化、懒加载，不是稳态 | capture range 只抓测量步（nsys 用 capture-range，ncu 用 `--profile-from-start off`） |
| 用 profile 运行的时间当作性能 | nsys 有开销、每阶段同步，ncu 更是重放加锁频 | 时间用 12.2 的 benchmark，nsys 只看构成，ncu 只看指标 |
| 拿 ncu 的 duration 与 nsys 比 | 数字对不上，误以为有 bug | 不比，ncu 只看利用率与比例 |
| 发射受限的问题去用 ncu | 单个 kernel 很健康，却找不到问题 | 先看 12.7 的 GPU 利用率自检，低就回到 nsys 时间线 |
| ncu 不加过滤 | 慢到跑不完 | `-k`、`-s`、`-c`，先 `-c 1` |
| NVTX 区间内没同步 | CPU 区间极短，被误读 | 阶段末尾 `synchronize`（仅 profile 模式） |
| backward 归因为空 | 误以为反向没有 kernel | 12.8 的验证与备选 |
| 步数太多 | nsys 文件巨大，GUI 卡 | nsys 3 步 |
| `other` 类放着不管 | 占比失真 | 逐个看最大的 `other` 并补规则 |
| 标注粒度太细 | CPU 变慢，小模型尤其明显 | 计时不开细标注 |
| GUI 版本比生成工具旧 | 打不开 | 用不低于 nsys、ncu 的 GUI |
| ncu 权限报错 | 无法采集 | 联系管理员放开计数器权限 |
| 卡不空闲 | 数据不可信 | 运行前 `nvidia-smi` |

## 12.13 作业问题与取数方式对照（3.4 与 3.6）

| 问题 | 取数方式 |
|---|---|
| forward 中耗时最长的 kernel，调用次数 | `nvtx_kern_sum` 或时间窗法，取 `forward` 区间 |
| 同一个 kernel 在 forward 与 forward+backward 中的占比 | 两个区间各算一次占比 |
| 除 matmul 外占时间最多的 kernel | 类别表里除 `gemm` 之外最大的类别 |
| 完整训练步里 matmul 的占比 | 类别表（含 backward 与 optimizer） |
| attention 内 softmax 与 matmul 的时间比，对比 FLOPs 比 | `attn.scores`、`attn.softmax`、`attn.av` 三个区间 |
| 为什么 softmax 时间占比远高于它的 FLOPs 占比 | ncu 看 softmax kernel 的 Memory 与 Compute Throughput（12.10） |
| 加了 `torch.compile` 之后变了什么 | 重新 profile，对比 kernel 数与类别表；融合后的 kernel 可再用 ncu 看利用率 |

## 12.14 回填与解释

按 11.5 的清单逐行回填，预测值直接抄自 11.5，不要在这里另起一套数。

| 来源 | 预测项（11.5） | 实测来源 |
|---|---|---|
| (1) | forward、backward、optimizer 时间 | 12.2 的 benchmark |
| (2) | 注意力、FFN、其余的时间占比 | `attn.*`、`ffn` 区间（`nvtx_kern_sum`） |
| (3) | 各类别 kernel 数与时间占比 | 12.7 的类别表 |
| (4) | 各区间 kernel 数与时间 | `nvtx_kern_sum` 或时间窗法 |
| 显存 | 峰值 | 12.11 的 `max_memory_allocated` |
| "受限类型"类结论 | 访存受限、算力受限 | 12.10 的 ncu 指标 |

回填模板（每项一行）：

| 项 | 预测 | 实测 | 实测 ÷ 预测 | 差异原因 | ncu 证据 |
|---|---|---|---|---|---|

"ncu 证据"列只对需要解释的差异填，写具体指标，例如"softmax：Memory 约 80%，Compute 约 10%，带宽受限"，而不是写"推断为访存受限"。没抓 ncu 的行留空。

**差异怎么解释**：比值在 0.8 到 1.25 之间视为预测成立，超出就要找原因。

| 现象 | 常见原因 | 用什么确认 |
|---|---|---|
| 实测时间远高于预测，GPU 利用率低 | 发射受限：kernel 太小太多，CPU 追不上（小模型、短 ctx、逐参数 optimizer） | nsys 的利用率自检与时间线空隙 |
| 实测 kernel 数多于预测 | 估计漏了类型转换、`contiguous`、`cat` 等隐藏拷贝；autograd 额外的 kernel | nsys 的 kernel 清单 |
| 注意力时间占比高于 FLOPs 占比 | 访存密集，$T^2$ 大小的中间张量读写 | ncu：softmax、缩放、mask 的 Memory 高、Compute 低 |
| GEMM 时间比理想值慢 | 形状小、强度低，达不到峰值 $P$；TF32 没关或 dtype 没统一 | ncu：Compute Throughput、occupancy、kernel 名里是否有 tf32 |
| 逐元素 kernel 时间占比大 | 访存受限，每个 kernel 都要读写整个张量 | ncu：Memory Throughput 接近带宽峰值 |
| backward 时间不是 forward 的 2 倍 | 访存部分的倍数估错；或归因窗口没对齐（12.8） | 时间窗法重新归因 |
| 显存实测高于预估 | 份数估少、临时 buffer、allocator 碎片 | `max_memory_reserved` 与快照 |
| 显存实测低于预估 | 某些张量未被保存（融合、重算），或 $c$ 取大了 | `saved_tensors_hooks` |

最后把"最出乎意料的三项"写成注解：哪项预测错了、实测是什么、原因是什么（带证据）、下一步优化什么。这三条比整张表更重要。

---
